In [1]:
from pathlib import Path

messages_dir = Path.cwd().resolve().parent / 'tests' / 'messages'
message_files = sorted(messages_dir.glob('msg*.txt'))
messages_list = []
globals().update(
    {
        file_path.stem: file_path.read_text(encoding='utf-8')
        for file_path in message_files
    }
)

for file_path in message_files:
    messages_list.append(file_path.stem)



for file_path in message_files:
    print(f"Loaded {file_path.name} into variable {file_path.stem}")


Loaded msg1.txt into variable msg1
Loaded msg2.txt into variable msg2
Loaded msg3.txt into variable msg3
Loaded msg4.txt into variable msg4
Loaded msg5.txt into variable msg5
Loaded msg6.txt into variable msg6
Loaded msg7.txt into variable msg7
Loaded msg8.txt into variable msg8


In [2]:
print(messages_list)

['msg1', 'msg2', 'msg3', 'msg4', 'msg5', 'msg6', 'msg7', 'msg8']


In [3]:
import sys,os
sys.path.append(os.path.abspath('..'))

sys.path.append(os.path.abspath('.'))

In [4]:
from tools.extractJSON import extract_expense_test,extract_info_meter_sheet_test,extract_info_electronic_sales_sheet_test,rewrite_report_for_llm

In [5]:
from expected.msg1 import fuel
print(fuel)

{'date': '22/06/2026', 'pumps': {'PMS 1': {'opening': 1709211.595, 'closing': 679761.213}, 'PMS 2': {'opening': 1396086.013, 'closing': 348384.545}, 'AGO 1': {'opening': 857618.798, 'closing': 549370.32}, 'AGO 2': {'opening': 449227.617, 'closing': 294349.184}}, 'rtt': {'PMS': 9.861, 'AGO': 8.812}}


In [6]:
import importlib
expected_outputs = {}

In [7]:
for msg in messages_list:
    module = importlib.import_module(f"expected.{msg}")
    fuel = module.fuel
    electronic_sales = module.electronic_sales
    expenses = module.expenses
    expected_outputs[msg] = {'fuel': fuel, 'electronic_sales': electronic_sales, 'expenses': expenses}

In [8]:
def evalaute_fuel(output:dict, expected: dict):
    expected_extracted_keys = set(expected.keys()).union(set(expected["pumps"].keys())).union(set(expected["rtt"].keys()))
    expected_meter_keys = set()
    nozzle_names = list(expected["pumps"].keys())
    for nozzle in nozzle_names:
        name = f"{nozzle}:opening"
        name1 = f"{nozzle}:closing"
        expected_meter_keys.add(name)
        expected_meter_keys.add(name1)
    total = len(expected_extracted_keys) + len(expected_meter_keys)
    found = 0
    for key in output.keys():
        if key in expected_extracted_keys:
            found +=1

    try:
        pump_keys = set(output["pumps"].keys())
        for pump_key in pump_keys:
            name = f"{pump_key}:opening"
            name1 = f"{pump_key}:closing"
            if name in expected_meter_keys:
                found += 1
            if name1 in expected_meter_keys:
                found += 1
            if pump_key in expected_extracted_keys:
                found += 1
    except KeyError:
        pass

    #checking rtts
    try:
        rtt_keys = set(output["rtt"].keys())
        for rtt_key in rtt_keys:
            if rtt_key in expected_extracted_keys:
                found += 1
    except KeyError:
        pass
    recall = found/total

    # Calculate expected_accurate dynamically
    expected_accurate = 1  # for date
    try:
        expected_accurate += len(expected["rtt"])  # number of RTT entries
    except KeyError:
        pass
    try:
        expected_accurate += len(expected["pumps"]) * 2  # opening + closing for each pump
    except KeyError:
        pass
    
    actual_accurate = 0
    #checkign for date
    if output["date"] == expected["date"]:
        actual_accurate += 1
    #checking rtt
    try:
        rtt_keys = set(output["rtt"].keys())
        for rtt_key in rtt_keys:
            if output["rtt"][rtt_key] == expected["rtt"][rtt_key]:
                actual_accurate += 1
    except KeyError:
        pass
    #checking meter readings
    try:
        nozzle_names = list(expected["pumps"].keys())
        for nozzle in nozzle_names:
            try:
                if output["pumps"][nozzle]["opening"] == expected["pumps"][nozzle]["opening"]:
                    actual_accurate += 1
            except KeyError:
                pass
            try: 
                if output["pumps"][nozzle]["closing"] == expected["pumps"][nozzle]["closing"]:
                    actual_accurate += 1
            except KeyError:
                pass
    except KeyError:
        pass
    accuracy = actual_accurate/expected_accurate
    
    return recall, accuracy


In [9]:
print(evalaute_fuel(expected_outputs["msg1"]["fuel"], expected_outputs["msg1"]["fuel"]))

(1.0, 1.0)


In [10]:
def evalaute_electronic_sales(output: dict, expected: dict):
    expected_date = expected.get("date")
    expected_values = expected.get("electronic_sales", {})

    output_section = output
    output_date = output_section.get("date")
    output_values = output_section.get("electronic_sales", {}) if isinstance(output_section, dict) else {}

    total = len(expected_values) + 1
    found = 0
    accurate = 0

    if output_date == expected_date:
        found += 1
        accurate += 1
    elif output_date is not None:
        found += 1

    for key, expected_amount in expected_values.items():
        if key in output_values:
            found += 1
            if output_values[key] == expected_amount:
                accurate += 1

    recall = found/total
    accuracy = accurate/total
    return recall, accuracy


In [11]:
print(expected_outputs["msg1"]["electronic_sales"])

{'date': '22/06/2026', 'electronic_sales': {'MOMOPAY': None, 'AIRTEL': None, 'VISA CARD': 130000, 'RUBIS CARD': 31000, 'RUBIS APP': None}}


In [12]:
def evalaute_expenses(output: dict, expected: dict):
    expected_date = expected.get("date")
    expected_items = expected.get("expenses", [])
    expected_by_name = {item["expense_name"]: item["amount"] for item in expected_items}

    output_section = output
    output_date = output_section.get("date")
    output_items = output_section.get("expenses", []) if isinstance(output_section, dict) else []

    output_by_name = {
        item.get("expense_name"): item.get("amount")
        for item in output_items
        if isinstance(item, dict) and "expense_name" in item
    }

    total = len(expected_by_name)  + 1
    found = 0
    accurate = 0

    if output_date == expected_date:
        found += 1
        accurate += 1
    elif output_date is not None:
        found += 1

    for name, expected_amount in expected_by_name.items():
        if name in output_by_name:
            found += 1
            if output_by_name[name] == expected_amount:
                accurate += 1
        

    recall = found/total
    accuracy = accurate/total
    return recall, accuracy


In [13]:
recalls = []
accuracies = []
import time

In [14]:
raw_messages = {
    "msg1": msg1,
    "msg2": msg2,
    "msg3": msg3,
    "msg4": msg4,
    "msg5": msg5,
    "msg6": msg6,
    "msg7": msg7,
    "msg8": msg8,
}

In [15]:
def evaluate(start,end):
    for message in messages_list[start:end]:
        report_text = raw_messages[message]   # "msg1" → the real report text

        extracted_fuel = extract_info_meter_sheet_test(report_text)
        groundtruth_fuel = expected_outputs[message]["fuel"]
        time.sleep(10)

        extracted_expenses = extract_expense_test(report_text)
        groundtruth_expenses = expected_outputs[message]["expenses"]
        time.sleep(10)


        extracted_electronic_sales = extract_info_electronic_sales_sheet_test(report_text)
        groundtruth_electronics = expected_outputs[message]["electronic_sales"]
        time.sleep(10)


        recall_fuel, accuracy_fuel = evalaute_fuel(extracted_fuel, groundtruth_fuel)
        recall_expenses, accuracy_expenses = evalaute_expenses(extracted_expenses, groundtruth_expenses)
        recall_electronics, accuracy_electronics = evalaute_electronic_sales(extracted_electronic_sales, groundtruth_electronics)

        recalls.extend([recall_fuel, recall_expenses, recall_electronics])
        accuracies.extend([accuracy_fuel, accuracy_expenses, accuracy_electronics])

        print(f"""
        {'=' * 60}
        📋 RESULTS FOR: {message}
        {'=' * 60}

        ⛽ Fuel
            recall:   {recall_fuel:.2%}
            accuracy: {accuracy_fuel:.2%}

        💰 Expenses
            recall:   {recall_expenses:.2%}
            accuracy: {accuracy_expenses:.2%}

        🔌 Electronic Sales
            recall:   {recall_electronics:.2%}
            accuracy: {accuracy_electronics:.2%}

        {'=' * 60}
        """)
        time.sleep(10)
        

In [16]:
evaluate(0,3)


        📋 RESULTS FOR: msg1

        ⛽ Fuel
            recall:   100.00%
            accuracy: 90.91%

        💰 Expenses
            recall:   100.00%
            accuracy: 100.00%

        🔌 Electronic Sales
            recall:   100.00%
            accuracy: 100.00%

        

        📋 RESULTS FOR: msg2

        ⛽ Fuel
            recall:   100.00%
            accuracy: 70.59%

        💰 Expenses
            recall:   100.00%
            accuracy: 100.00%

        🔌 Electronic Sales
            recall:   100.00%
            accuracy: 66.67%

        
Error parsing JSON: Expecting ',' delimiter: line 2 column 13 (char 14)
Cleaned response: {
  "date": 20/04/2026,
  "pumps": {
    "PMS 1": {
      "opening": 1660801.026,
      "closing": 655635.428
    },
    "PMS 2": {
      "opening": 1357057.137,
      "closing": 333539.145
    },
    "PMS 3": {
      "opening": null,
      "closing": null
    },
    "PMS 4": {
      "opening": null,
      "closing": null
    },
    "AGO 1": {
 

JSONDecodeError: Expecting ',' delimiter: line 2 column 13 (char 14)

In [ ]:
print(accuracies)
print(recalls)

[0.7272727272727273, 1.0, 1.0, 0.7058823529411765, 1.0, 0.8333333333333334, 0.7647058823529411, 0.8333333333333334, 0.8333333333333334]
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.8333333333333334, 1.0]


In [ ]:
evaluate(3,7)


        📋 RESULTS FOR: msg4

        ⛽ Fuel
            recall:   100.00%
            accuracy: 70.59%

        💰 Expenses
            recall:   80.00%
            accuracy: 80.00%

        🔌 Electronic Sales
            recall:   100.00%
            accuracy: 100.00%

        

        📋 RESULTS FOR: msg5

        ⛽ Fuel
            recall:   100.00%
            accuracy: 76.47%

        💰 Expenses
            recall:   100.00%
            accuracy: 100.00%

        🔌 Electronic Sales
            recall:   100.00%
            accuracy: 100.00%

        

        📋 RESULTS FOR: msg6

        ⛽ Fuel
            recall:   100.00%
            accuracy: 76.47%

        💰 Expenses
            recall:   100.00%
            accuracy: 100.00%

        🔌 Electronic Sales
            recall:   83.33%
            accuracy: 50.00%

        

        📋 RESULTS FOR: msg7

        ⛽ Fuel
            recall:   100.00%
            accuracy: 76.47%

        💰 Expenses
            recall:   100.00%
    

In [ ]:
print(accuracies)
print(recalls)

[0.7272727272727273, 1.0, 1.0, 0.7058823529411765, 1.0, 0.8333333333333334, 0.7647058823529411, 0.8333333333333334, 0.8333333333333334]
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.8333333333333334, 1.0]


In [ ]:
accuracy = sum(accuracies)/len(accuracies)
recall = sum(recalls)/len(recalls)

In [ ]:
print(accuracy)

0.8570409982174687


In [ ]:
print(recall)

0.9746031746031747


So the current accuracy stands at 85.7% and the recall is 97.4%. In order to try and improve it I will try to use another llm to rewrite the report before passing it to the parser. Hopefully this works.